# Sujet 3 — Analyse mondiale des jeux vidéo

**Projet Python pour la Data — INSI**

**Membres du groupe (Groupe 1 L1D) :**

| N° | Nom et Prénom | Matricule |
|----|--------------------------------------------|-----------|
| 1 | HARIMINO Faly Henintsoa | 401 |
| 2 | ANDRIAMIRADO Fanomezantsoa Fiononana | 402 |
| 3 | (à compléter) | ... |
| 4 | (à compléter) | ... |

**Répartition du travail sur ce sujet :** voir `README.txt`.

## Présentation du dataset

Le dataset **Video Game Sales** (Kaggle, `vgsales.csv`) recense 16 598 jeux vidéo commercialisés dans le monde, avec pour chacun : plateforme, année de sortie, genre, éditeur, et les ventes en millions d'unités par zone géographique (Amérique du Nord, Europe, Japon, autres régions) ainsi que les ventes mondiales totales.

## Problématique

Une entreprise du secteur souhaite comprendre les tendances historiques du marché mondial du jeu vidéo : évolution dans le temps, poids des différentes zones géographiques, plateformes et éditeurs dominants. L'analyse est réalisée uniquement avec NumPy et Pandas.

## Importation des bibliothèques

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## Chargement des données

In [2]:
df = pd.read_csv('data/vgsales.csv')
print('Dataset chargé avec succès.')

Dataset chargé avec succès.


## Partie A — Exploration

In [3]:
# 1-2. Premières et dernières lignes
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [4]:
df.tail()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
16593,16596,Woody Woodpecker in Crazy Castle 5,GBA,2002.0,Platform,Kemco,0.01,0.00,0.0,0.0,0.01
16594,16597,Men in Black II: Alien Escape,GC,2003.0,Shooter,Infogrames,0.01,0.00,0.0,0.0,0.01
16595,16598,SCORE International Baja 1000: The Official Game,PS2,2008.0,Racing,Activision,0.00,0.00,0.0,0.0,0.01
16596,16599,Know How 2,DS,2010.0,Puzzle,7G//AMES,0.00,0.01,0.0,0.0,0.01
16597,16600,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.0,0.0,0.01


In [5]:
# 3. Dimensions
print('Dimensions :', df.shape)

Dimensions : (16598, 11)


In [6]:
# 4. Types de variables
df.dtypes

Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

In [7]:
# 5. Valeurs manquantes
manquants = df.isnull().sum()
manquants[manquants > 0]

Year         271
Publisher     58
dtype: int64

In [8]:
# 6. Doublons
print('Nombre de doublons :', df.duplicated().sum())

Nombre de doublons : 0


In [9]:
# 7. Statistiques descriptives
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


## Partie B — Nettoyage

In [10]:
# Year : 271 valeurs manquantes. On ne peut pas deviner une année de sortie -> on ne l'invente pas.
# On convertit Year en Int64 (entier nullable) plutôt que float, et on garde les lignes en indiquant
# clairement les jeux à année inconnue (nécessaire pour les analyses temporelles, exclus quand la Partie F l'exige).
df['Year'] = df['Year'].astype('Int64')
print('Valeurs manquantes pour Year :', df['Year'].isnull().sum())
print('Plage des années connues :', df['Year'].min(), '-', df['Year'].max())

Valeurs manquantes pour Year : 271
Plage des années connues : 1980 - 2020


In [11]:
# Publisher : 58 valeurs manquantes -> remplacées par 'Unknown' (on ne peut pas deviner l'éditeur,
# mais on ne veut pas perdre les lignes, qui restent valides pour les autres analyses)
df['Publisher'] = df['Publisher'].fillna('Unknown')
print('Valeurs manquantes pour Publisher :', df['Publisher'].isnull().sum())

Valeurs manquantes pour Publisher : 0


In [12]:
# Vérification des colonnes numériques de ventes : pas de valeur négative ou aberrante attendue
ventes_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
print(df[ventes_cols].describe())
print('Valeurs négatives détectées :', (df[ventes_cols] < 0).sum().sum())

           NA_Sales      EU_Sales      JP_Sales   Other_Sales  Global_Sales
count  16598.000000  16598.000000  16598.000000  16598.000000  16598.000000
mean       0.264667      0.146652      0.077782      0.048063      0.537441
std        0.816683      0.505351      0.309291      0.188588      1.555028
min        0.000000      0.000000      0.000000      0.000000      0.010000
25%        0.000000      0.000000      0.000000      0.000000      0.060000
50%        0.080000      0.020000      0.000000      0.010000      0.170000
75%        0.240000      0.110000      0.040000      0.040000      0.470000
max       41.490000     29.020000     10.220000     10.570000     82.740000
Valeurs négatives détectées : 0


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  Int64  
 4   Genre         16598 non-null  str    
 5   Publisher     16598 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: Int64(1), float64(5), int64(1), str(4)
memory usage: 1.4 MB


**Transformations réalisées :** `Year` est converti en entier nullable (`Int64`) pour un format cohérent (une année ne doit pas être un flottant) tout en conservant les valeurs manquantes plutôt que de les supprimer ou de les inventer. `Publisher` manquant est remplacé par `'Unknown'` pour ne pas perdre de lignes exploitables. Les colonnes de ventes ne contiennent aucune valeur négative ni doublon, aucune correction n'y est nécessaire.

## Partie C — Vérification avec NumPy

In [14]:
# 1-2. Calculated_Global_Sales et Difference
df['Calculated_Global_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']
df['Difference'] = np.round(df['Global_Sales'] - df['Calculated_Global_Sales'], 4)
df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Calculated_Global_Sales', 'Difference']].head()

,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Calculated_Global_Sales,Difference
0,41.49,29.02,3.77,8.46,82.74,82.74,-0.00
1,29.08,3.58,6.81,0.77,40.24,40.24,0.00
2,15.85,12.88,3.79,3.31,35.82,35.83,-0.01
3,15.75,11.01,3.28,2.96,33.00,33.00,0.00
4,11.27,8.89,10.22,1.00,31.37,31.38,-0.01


In [15]:
# 3. Analyse des différences
print(df['Difference'].describe())
nb_incoherent = (np.abs(df['Difference']) > 0.01).sum()
print(f"Nombre de lignes avec un écart > 0.01 : {nb_incoherent} sur {len(df)} ({nb_incoherent/len(df)*100:.2f}%)")

count    16598.000000
mean         0.000277
std          0.005223
min         -0.020000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.020000
Name: Difference, dtype: float64
Nombre de lignes avec un écart > 0.01 : 10 sur 16598 (0.06%)


**Interprétation :** l'écart entre `Global_Sales` et la somme des ventes régionales est quasi nul pour l'immense majorité des jeux (arrondis de saisie), ce qui confirme que `Global_Sales` correspond bien à la somme des quatre zones et valide la cohérence interne du dataset.

## Partie D — Analyse mondiale

In [16]:
# 1. Top 10 jeux les plus vendus au monde
top10_jeux = df.nlargest(10, 'Global_Sales')[['Name', 'Platform', 'Year', 'Global_Sales']]
top10_jeux

,Name,Platform,Year,Global_Sales
0,Wii Sports,Wii,2006,82.74
1,Super Mario Bros.,NES,1985,40.24
2,Mario Kart Wii,Wii,2008,35.82
3,Wii Sports Resort,Wii,2009,33.00
4,Pokemon Red/Pokemon Blue,GB,1996,31.37
5,Tetris,GB,1989,30.26
6,New Super Mario Bros.,DS,2006,30.01
7,Wii Play,Wii,2006,29.02
8,New Super Mario Bros. Wii,Wii,2009,28.62
9,Duck Hunt,NES,1984,28.31


In [17]:
# 2. Top 10 plateformes par ventes mondiales
top10_plateformes = df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10)
top10_plateformes.round(2)

Platform
PS2     1255.64
X360     979.96
PS3      957.84
Wii      926.71
DS       822.49
PS       730.66
GBA      318.50
PSP      296.28
PS4      278.10
PC       258.82
Name: Global_Sales, dtype: float64

In [18]:
# 3. Top 10 éditeurs par ventes mondiales
top10_editeurs = df.groupby('Publisher')['Global_Sales'].sum().sort_values(ascending=False).head(10)
top10_editeurs.round(2)

Publisher
Nintendo                        1786.56
Electronic Arts                 1110.32
Activision                       727.46
Sony Computer Entertainment      607.50
Ubisoft                          474.72
Take-Two Interactive             399.54
THQ                              340.77
Konami Digital Entertainment     283.64
Sega                             272.99
Namco Bandai Games               254.09
Name: Global_Sales, dtype: float64

In [19]:
# 4. Genres les plus vendus
ventes_genre = df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False)
ventes_genre.round(2)

Genre
Action          1751.18
Sports          1330.93
Shooter         1037.37
Role-Playing     927.37
Platform         831.37
Misc             809.96
Racing           732.04
Fighting         448.91
Simulation       392.20
Puzzle           244.95
Adventure        239.04
Strategy         175.12
Name: Global_Sales, dtype: float64

In [20]:
# 5. Ventes mondiales moyennes par jeu
ventes_moyennes = df['Global_Sales'].mean()
print(f"Ventes mondiales moyennes par jeu : {ventes_moyennes:.3f} millions d'unités")

Ventes mondiales moyennes par jeu : 0.537 millions d'unités


## Partie E — Analyse géographique

In [21]:
# Comparaison des 4 zones géographiques (total des ventes)
comparaison_zones = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum().sort_values(ascending=False)
comparaison_zones.round(2)

NA_Sales       4392.95
EU_Sales       2434.13
JP_Sales       1291.02
Other_Sales     797.75
dtype: float64

In [22]:
# Genre le plus populaire dans chaque zone
genre_par_zone = {}
for zone in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']:
    top_genre = df.groupby('Genre')[zone].sum().idxmax()
    genre_par_zone[zone] = top_genre
pd.Series(genre_par_zone, name='genre_dominant')

NA_Sales             Action
EU_Sales             Action
JP_Sales       Role-Playing
Other_Sales          Action
Name: genre_dominant, dtype: str

In [23]:
# Plateformes les plus importantes par zone
plateformes_par_zone = {}
for zone in ['NA_Sales', 'EU_Sales', 'JP_Sales']:
    top3 = df.groupby('Platform')[zone].sum().sort_values(ascending=False).head(3)
    plateformes_par_zone[zone] = list(top3.index)
plateformes_par_zone

{'NA_Sales': ['X360', 'PS2', 'Wii'],
 'EU_Sales': ['PS3', 'PS2', 'X360'],
 'JP_Sales': ['DS', 'PS', 'PS2']}

**Interprétation :** le Japon se distingue nettement des autres zones (préférence marquée pour certains genres et plateformes typiquement japonaises), tandis que l'Amérique du Nord et l'Europe partagent des tendances plus proches, portées par les mêmes grandes plateformes occidentales. Cela illustre des marchés culturellement différenciés.

## Partie F — Analyse temporelle

In [24]:
df_annee = df.dropna(subset=['Year']).copy()
df_annee['Year'] = df_annee['Year'].astype(int)

# 1. Ventes mondiales annuelles
ventes_par_annee = df_annee.groupby('Year')['Global_Sales'].sum()
ventes_par_annee.tail(10)

Year
2009    667.30
2010    600.45
2011    515.99
2012    363.54
2013    368.11
2014    337.05
2015    264.44
2016     70.93
2017      0.05
2020      0.29
Name: Global_Sales, dtype: float64

In [25]:
# 2. Année ayant enregistré le plus de ventes
annee_top = ventes_par_annee.idxmax()
print(f"Année avec le plus de ventes mondiales : {annee_top} ({ventes_par_annee.max():.2f} millions d'unités)")

Année avec le plus de ventes mondiales : 2008 (678.90 millions d'unités)


In [26]:
# 3. Nombre de jeux publiés chaque année
jeux_par_annee = df_annee.groupby('Year').size()
jeux_par_annee.tail(10)

Year
2009    1431
2010    1259
2011    1139
2012     657
2013     546
2014     582
2015     614
2016     344
2017       3
2020       1
dtype: int64

In [27]:
# 4. Ventes moyennes par jeu et par année
ventes_moy_par_annee = df_annee.groupby('Year')['Global_Sales'].mean()
ventes_moy_par_annee.tail(10).round(3)

Year
2009    0.466
2010    0.477
2011    0.453
2012    0.553
2013    0.674
2014    0.579
2015    0.431
2016    0.206
2017    0.017
2020    0.290
Name: Global_Sales, dtype: float64

In [28]:
# Comparaison par période : avant 2000 / 2000-2009 / 2010 et après
conditions = [
    df_annee['Year'] < 2000,
    (df_annee['Year'] >= 2000) & (df_annee['Year'] <= 2009),
    df_annee['Year'] >= 2010,
]
choix = ['Avant 2000', '2000-2009', '2010 et après']
df_annee['Periode'] = np.select(conditions, choix, default='Inconnue')

comparaison_periodes = df_annee.groupby('Periode').agg(
    nb_jeux=('Name', 'count'),
    ventes_totales=('Global_Sales', 'sum'),
    ventes_moyennes=('Global_Sales', 'mean'),
).round(3)
comparaison_periodes = comparaison_periodes.reindex(choix)
comparaison_periodes

,nb_jeux,ventes_totales,ventes_moyennes
Periode,,,
Avant 2000,1974,1655.49,0.839
2000-2009,9208,4644.02,0.504
2010 et après,5145,2520.85,0.490


## Partie G — Conclusion

In [29]:
print(f"Jeu le plus vendu au monde : {top10_jeux.iloc[0]['Name']} ({top10_jeux.iloc[0]['Global_Sales']} M)")
print(f"Plateforme n°1 : {top10_plateformes.index[0]}")
print(f"Éditeur n°1 : {top10_editeurs.index[0]}")
print(f"Genre n°1 : {ventes_genre.index[0]}")
print(f"Année record : {annee_top}")

Jeu le plus vendu au monde : Wii Sports (82.74 M)
Plateforme n°1 : PS2
Éditeur n°1 : Nintendo
Genre n°1 : Action
Année record : 2008


**Conclusions et observations majeures :**

1. **Un très petit nombre de jeux « phénomènes » concentre une part importante des ventes mondiales** (le top 10 des jeux les plus vendus dépasse largement la moyenne générale), révélant un marché où le succès est fortement concentré sur quelques licences majeures.
2. **Nintendo et les plateformes Nintendo dominent historiquement le classement**, en particulier sur le marché japonais, ce qui traduit une stratégie de plateformes propriétaires très efficace sur le long terme.
3. **L'Amérique du Nord reste, sur l'ensemble de la période, la plus grande zone de vente en volume**, suivie de l'Europe, tandis que le Japon constitue un marché plus petit mais avec des préférences de genres/plateformes très spécifiques (RPG, plateformes locales).
4. **Le marché a connu un pic d'activité au cours des années 2000**, porté par l'essor des consoles de salon et portables de l'époque (ex. PS2, DS, Wii), avant un ralentissement progressif du nombre de titres physiques recensés dans la dernière décennie du dataset.
5. **La comparaison par période (avant 2000 / 2000-2009 / 2010+) montre une croissance puis une stabilisation ou un recul du nombre de sorties et des ventes totales**, cohérente avec la transition du marché vers le numérique et le mobile, moins bien capturée par ce type de dataset orienté ventes physiques.
6. **Genre Action et Sports figurent parmi les catégories les plus vendues mondialement**, alors que leur popularité relative varie sensiblement d'une zone géographique à l'autre, confirmant l'intérêt d'une stratégie commerciale différenciée par région plutôt qu'une approche uniforme au niveau mondial.